In [ ]:
from pathlib import Path
from typing import Dict, Optional, Sequence, Tuple
import warnings

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings("ignore")

# =========================================================
# 0) CONFIGURATION
# =========================================================
PROJECT_ROOT_CANDIDATES = [
    Path("Data_Science_Midterm_Project-main"),
    Path("./Data_Science_Midterm_Project-main"),
    Path("../Data_Science_Midterm_Project-main"),
]

MERGED_SENTIMENT_FILE_REL = Path("sentiment_data_processed/model_ready_quarterly_with_scaled.csv")
MACRO_BASELINE_FILE_REL = Path("secondary_data_processed.csv")

OUTPUT_DIR = Path("branch2_outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

DATE_COL = "quarter_end"
SENTIMENT_REAL_START = "2019-09-30"
TRAIN_END = "2022-12-31"
FORECAST_HORIZONS = [1, 2, 4]
BREAK_DATES = ["2021-12-31", "2022-03-31", "2023-03-31"]

TARGET_MACRO = "gdp_qoq_pct_recalc"
TARGET_OVERLAPS = ["gdp_growth", "gdp_qoq_pct_recalc"]

MACRO_EXOG = ["inflation", "interest_rate"]
SENTIMENT_MODEL_VARS = ["net_sentiment", "avg_sentiment"]
SENTIMENT_AUDIT_VARS = [
    "n_comments",
    "avg_sentiment",
    "net_sentiment",
    "positive_ratio",
    "negative_ratio",
]
REAL_DATA_FLAG_COL = "was_missing_period"

ORDER_MAP = {
    "gdp_qoq_pct_recalc": {
        "macro_final": (0, 0, 3),
        "macro_overlap": (1, 0, 0),
        "macro_sent_overlap": (1, 0, 0),
    },
    "gdp_growth": {
        "macro_overlap": (2, 0, 0),
        "macro_sent_overlap": (2, 0, 0),
    },
}


# =========================================================
# 1) GENERAL UTILITY FUNCTIONS
# =========================================================
def find_project_root(candidates: Sequence[Path]) -> Path:
    for c in candidates:
        if c.exists() and c.is_dir():
            return c.resolve()
    raise FileNotFoundError(
        "Could not find the folder 'Data_Science_Midterm_Project-main'. "
        "Please place the notebook/script in the same directory level as the extracted folder."
    )


def mae(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def mse(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean((y_true - y_pred) ** 2))


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mse(y_true, y_pred)))


def safe_mape(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def aicc_from_result(res) -> float:
    n = int(res.nobs)
    k = int(len(res.params))
    if n - k - 1 <= 0:
        return np.nan
    return float(res.aic + (2 * k * (k + 1)) / (n - k - 1))


def coefficient_table(res) -> pd.DataFrame:
    params = np.asarray(res.params)
    bse = np.asarray(res.bse)
    tvalues = np.asarray(res.tvalues)
    pvalues = np.asarray(res.pvalues)

    ci_raw = res.conf_int()
    ci = ci_raw.values if isinstance(ci_raw, pd.DataFrame) else np.asarray(ci_raw)

    if hasattr(res, "param_names") and res.param_names is not None:
        names = list(res.param_names)
    elif hasattr(res.model, "param_names") and res.model.param_names is not None:
        names = list(res.model.param_names)
    elif hasattr(res.params, "index"):
        names = list(res.params.index)
    else:
        names = [f"param_{i}" for i in range(len(params))]

    return pd.DataFrame(
        {
            "variable": names,
            "coef": params,
            "std_err": bse,
            "t_or_z": tvalues,
            "p_value": pvalues,
            "ci_low": ci[:, 0],
            "ci_high": ci[:, 1],
        }
    )


def make_lag(df: pd.DataFrame, col: str, lag: int = 1) -> pd.Series:
    return df[col].shift(lag).rename(f"{col}_lag{lag}")


# =========================================================
# 2) LOAD DATA
# =========================================================
def load_sentiment_merged(project_root: Path) -> pd.DataFrame:
    file_path = project_root / MERGED_SENTIMENT_FILE_REL
    if not file_path.exists():
        raise FileNotFoundError(f"Could not find merged sentiment file: {file_path}")

    df = pd.read_csv(file_path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df = df.sort_values(DATE_COL).reset_index(drop=True)
    return df


def load_macro_baseline(project_root: Path) -> pd.DataFrame:
    file_path = project_root / MACRO_BASELINE_FILE_REL
    if not file_path.exists():
        raise FileNotFoundError(f"Could not find macro baseline file: {file_path}")

    df = pd.read_csv(file_path)
    rename_map = {
        "Unnamed: 0": "quarter_end",
        "GDP_Raw": "gdp_raw",
        "Inflation": "inflation",
        "Interest_Rate": "interest_rate",
        "FDI_pct_GDP": "fdi_pct_gdp",
        "Investment_pct_GDP": "investment_pct_gdp",
        "Export_pct_GDP": "export_pct_gdp",
        "Unemployment_Rate": "unemployment_rate",
        "Gov_Spending_pct_GDP": "gov_spending_pct_gdp",
        "GDP_Growth": "gdp_growth",
        "log_GDP": "log_gdp",
        "GDP_qoq_pct_recalc": "gdp_qoq_pct_recalc",
        "GDP_yoy_pct_recalc": "gdp_yoy_pct_recalc",
        "Inflation_lag1": "inflation_lag1",
        "Inflation_lag2": "inflation_lag2",
        "Interest_Rate_lag1": "interest_rate_lag1",
        "Interest_Rate_lag2": "interest_rate_lag2",
        "log_GDP_diff1": "log_gdp_diff1",
        "log_GDP_diff4": "log_gdp_diff4",
        "Inflation_scaled": "inflation_scaled",
        "Interest_Rate_scaled": "interest_rate_scaled",
        "FDI_pct_GDP_scaled": "fdi_pct_gdp_scaled",
        "Investment_pct_GDP_scaled": "investment_pct_gdp_scaled",
        "Export_pct_GDP_scaled": "export_pct_gdp_scaled",
        "Unemployment_Rate_scaled": "unemployment_rate_scaled",
        "Gov_Spending_pct_GDP_scaled": "gov_spending_pct_gdp_scaled",
    }
    df = df.rename(columns=rename_map)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df = df.sort_values(DATE_COL).reset_index(drop=True)
    return df


# =========================================================
# 3) CREATE TWO DATA BRANCHES
# =========================================================
def prepare_branch1_macro_df(macro_df: pd.DataFrame) -> pd.DataFrame:
    return macro_df.copy()


def prepare_branch2_overlap_df(merged_df: pd.DataFrame) -> pd.DataFrame:
    out = merged_df.copy()
    out = out[out[DATE_COL] >= pd.to_datetime(SENTIMENT_REAL_START)].copy()

    if REAL_DATA_FLAG_COL in out.columns:
        out = out[out[REAL_DATA_FLAG_COL].fillna(1) == 0].copy()

    out = out.sort_values(DATE_COL).reset_index(drop=True)
    return out


# =========================================================
# 4) LEAKAGE AUDIT
# =========================================================
def leakage_audit(df: pd.DataFrame, audit_cols: Sequence[str], cutoff: str) -> pd.DataFrame:
    cutoff_ts = pd.to_datetime(cutoff)
    pre = df[df[DATE_COL] < cutoff_ts].copy()

    rows = []
    for col in audit_cols:
        if col in pre.columns:
            rows.append(
                {
                    "variable": col,
                    "non_null_count_pre_cutoff": int(pre[col].notna().sum()),
                    "nunique_pre_cutoff": int(pre[col].nunique(dropna=True)),
                    "first_non_null_value": pre[col].dropna().iloc[0] if pre[col].notna().any() else np.nan,
                }
            )
    return pd.DataFrame(rows)


# =========================================================
# 5) STATIONARITY + VIF
# =========================================================
def adf_kpss_one(series: pd.Series) -> Dict[str, float]:
    s = series.dropna().astype(float)
    if len(s) < 8:
        return {
            "adf_stat": np.nan,
            "adf_p": np.nan,
            "kpss_stat": np.nan,
            "kpss_p": np.nan,
        }

    try:
        adf_stat, adf_p, *_ = adfuller(s, autolag="AIC")
    except Exception:
        adf_stat, adf_p = np.nan, np.nan

    try:
        kpss_stat, kpss_p, *_ = kpss(s, regression="c", nlags="auto")
    except Exception:
        kpss_stat, kpss_p = np.nan, np.nan

    return {
        "adf_stat": adf_stat,
        "adf_p": adf_p,
        "kpss_stat": kpss_stat,
        "kpss_p": kpss_p,
    }


def stationarity_table(df: pd.DataFrame, series_list: Sequence[str]) -> pd.DataFrame:
    rows = []
    for col in series_list:
        if col in df.columns:
            stats_out = adf_kpss_one(df[col])
            note = (
                "stationary"
                if (
                    pd.notna(stats_out["adf_p"])
                    and stats_out["adf_p"] < 0.05
                    and pd.notna(stats_out["kpss_p"])
                    and stats_out["kpss_p"] >= 0.05
                )
                else "check differencing / stability"
            )
            rows.append({"series": col, **stats_out, "note": note})
    return pd.DataFrame(rows)


def vif_table(df: pd.DataFrame, predictors: Sequence[str]) -> pd.DataFrame:
    use_cols = [c for c in predictors if c in df.columns]
    data = df[use_cols].dropna().copy()

    if data.empty:
        return pd.DataFrame(columns=["variable", "VIF"])

    data = sm.add_constant(data, has_constant="add")
    rows = []
    for i, col in enumerate(data.columns):
        try:
            vif_val = variance_inflation_factor(data.values, i)
        except Exception:
            vif_val = np.nan
        rows.append({"variable": col, "VIF": vif_val})
    return pd.DataFrame(rows)


# =========================================================
# 6) DYNAMIC REGRESSION
# =========================================================
def fit_dynamic_regression(
    df: pd.DataFrame,
    target: str,
    predictors: Sequence[str],
    hac_lags: int = 1,
):
    required = [target] + list(predictors)
    data = df[required].dropna().copy()

    X = sm.add_constant(data[predictors], has_constant="add")
    y = data[target]

    model = sm.OLS(y, X)
    res_plain = model.fit()
    res_hac = model.fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    return data, res_plain, res_hac


def nested_sentiment_tests(
    res_small_plain,
    res_big_plain,
    res_big_hac,
    sentiment_terms: Sequence[str],
) -> pd.DataFrame:
    f_stat, f_p, _ = res_big_plain.compare_f_test(res_small_plain)

    names = list(res_big_hac.params.index)
    R = np.zeros((len(sentiment_terms), len(names)))
    for i, term in enumerate(sentiment_terms):
        if term not in names:
            raise KeyError(f"Could not find term '{term}' in the larger model.")
        R[i, names.index(term)] = 1.0

    wald = res_big_hac.wald_test(R)

    return pd.DataFrame(
        {
            "test": ["Nested_F_test", "HAC_Wald_joint_sentiment"],
            "statistic": [float(np.asarray(f_stat).squeeze()), float(np.asarray(wald.statistic).squeeze())],
            "p_value": [float(np.asarray(f_p).squeeze()), float(np.asarray(wald.pvalue).squeeze())],
        }
    )


# =========================================================
# 7) CHOW TEST / STRUCTURAL BREAK
# =========================================================
def chow_test(
    df: pd.DataFrame,
    target: str,
    predictors: Sequence[str],
    break_date: str,
) -> Dict[str, float]:
    use_cols = [DATE_COL, target] + list(predictors)
    data = df[use_cols].dropna().copy().sort_values(DATE_COL)

    br = pd.to_datetime(break_date)
    left = data[data[DATE_COL] <= br].copy()
    right = data[data[DATE_COL] > br].copy()

    k = len(predictors) + 1
    n1, n2 = len(left), len(right)

    if n1 <= k or n2 <= k:
        return {
            "break_date": break_date,
            "F_stat": np.nan,
            "p_value": np.nan,
            "n1": n1,
            "n2": n2,
        }

    X_pooled = sm.add_constant(data[predictors], has_constant="add")
    y_pooled = data[target]
    pooled_res = sm.OLS(y_pooled, X_pooled).fit()
    rss_pooled = float(np.sum(pooled_res.resid ** 2))

    X1 = sm.add_constant(left[predictors], has_constant="add")
    y1 = left[target]
    res1 = sm.OLS(y1, X1).fit()
    rss1 = float(np.sum(res1.resid ** 2))

    X2 = sm.add_constant(right[predictors], has_constant="add")
    y2 = right[target]
    res2 = sm.OLS(y2, X2).fit()
    rss2 = float(np.sum(res2.resid ** 2))

    numerator = (rss_pooled - (rss1 + rss2)) / k
    denominator = (rss1 + rss2) / (n1 + n2 - 2 * k)

    if denominator <= 0:
        return {
            "break_date": break_date,
            "F_stat": np.nan,
            "p_value": np.nan,
            "n1": n1,
            "n2": n2,
        }

    f_stat = numerator / denominator
    p_val = 1 - stats.f.cdf(f_stat, k, n1 + n2 - 2 * k)

    return {
        "break_date": break_date,
        "F_stat": float(f_stat),
        "p_value": float(p_val),
        "n1": int(n1),
        "n2": int(n2),
    }


# =========================================================
# 8) SARIMAX + RESIDUAL DIAGNOSTICS
# =========================================================
def fit_sarimax_full(
    df: pd.DataFrame,
    target: str,
    order: Tuple[int, int, int],
    exog_cols: Optional[Sequence[str]] = None,
    trend: str = "c",
):
    use_cols = [DATE_COL, target] + (list(exog_cols) if exog_cols else [])
    data = df[use_cols].dropna().copy().sort_values(DATE_COL)

    data_idx = data.set_index(DATE_COL).copy()
    endog = data_idx[target].astype(float)
    exog = data_idx[list(exog_cols)].astype(float) if exog_cols else None

    model = SARIMAX(
        endog=endog,
        exog=exog,
        order=order,
        trend=trend,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    res = model.fit(disp=False)
    return data.reset_index(drop=True), res


def residual_diagnostics(res, lb_lags=(4, 8), arch_lags: int = 4) -> Dict[str, pd.DataFrame]:
    resid = pd.Series(res.resid).dropna()

    lb = acorr_ljungbox(resid, lags=list(lb_lags), return_df=True).reset_index().rename(columns={"index": "lag"})
    jb_stat, jb_p, skew, kurt = sm.stats.jarque_bera(resid)
    arch_stat, arch_p, _, _ = het_arch(resid, nlags=arch_lags)

    overall = pd.DataFrame(
        {
            "metric": ["Jarque_Bera", "ARCH_LM"],
            "statistic": [jb_stat, arch_stat],
            "p_value": [jb_p, arch_p],
            "skew": [skew, np.nan],
            "kurtosis": [kurt, np.nan],
        }
    )
    return {"ljung_box": lb, "overall": overall}


# =========================================================
# 9) ROLLING-ORIGIN BACKTEST
# =========================================================
def _initial_train_position(dates: pd.Series, train_end: str) -> int:
    train_end_ts = pd.to_datetime(train_end)
    idx = np.searchsorted(pd.to_datetime(dates).values, np.datetime64(train_end_ts), side="right")
    return int(idx)


def rolling_origin_backtest(
    df: pd.DataFrame,
    target: str,
    order: Tuple[int, int, int],
    exog_cols: Optional[Sequence[str]] = None,
    initial_train_end: str = TRAIN_END,
    horizons: Sequence[int] = (1,),
    trend: str = "c",
    model_name: str = "model",
) -> pd.DataFrame:
    use_cols = [DATE_COL, target] + (list(exog_cols) if exog_cols else [])
    data = df[use_cols].dropna().copy().sort_values(DATE_COL).reset_index(drop=True)

    if len(data) < 20:
        raise ValueError(f"Sample is too short for rolling backtest: {len(data)} rows.")

    start_pos = _initial_train_position(data[DATE_COL], initial_train_end)
    max_h = max(horizons)
    rows = []

    for origin_end in range(start_pos, len(data) - max_h + 1):
        train = data.iloc[:origin_end].copy()
        if len(train) < 12:
            continue

        train_idx = train.set_index(DATE_COL)
        endog_train = train_idx[target].astype(float)
        exog_train = train_idx[list(exog_cols)].astype(float) if exog_cols else None

        try:
            fitted = SARIMAX(
                endog=endog_train,
                exog=exog_train,
                order=order,
                trend=trend,
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(disp=False)
        except Exception as e:
            print(f"[WARN] Fit error | {model_name} | origin={train[DATE_COL].iloc[-1]} | {e}")
            continue

        for h in horizons:
            future = data.iloc[origin_end : origin_end + h].copy()
            if len(future) < h:
                continue

            future_idx = future.set_index(DATE_COL)
            exog_future = future_idx[list(exog_cols)].astype(float) if exog_cols else None

            try:
                fc = fitted.get_forecast(steps=h, exog=exog_future)
                pred_mean = fc.predicted_mean
                pred = float(pred_mean.iloc[-1] if hasattr(pred_mean, "iloc") else pred_mean[-1])

                ci80 = fc.conf_int(alpha=0.20)
                ci95 = fc.conf_int(alpha=0.05)

                ci80_arr = ci80.values if isinstance(ci80, pd.DataFrame) else np.asarray(ci80)
                ci95_arr = ci95.values if isinstance(ci95, pd.DataFrame) else np.asarray(ci95)
            except Exception as e:
                print(f"[WARN] Forecast error | {model_name} | horizon={h} | {e}")
                continue

            actual = float(future[target].iloc[-1])
            prev_actual = float(train[target].iloc[-1])

            rows.append(
                {
                    "model": model_name,
                    "target": target,
                    "origin_end": train[DATE_COL].iloc[-1],
                    "forecast_date": future[DATE_COL].iloc[-1],
                    "horizon": h,
                    "actual": actual,
                    "prediction": pred,
                    "prev_actual": prev_actual,
                    "error": pred - actual,
                    "abs_error": abs(pred - actual),
                    "ape": np.nan if actual == 0 else abs((pred - actual) / actual) * 100,
                    "pred_direction": np.sign(pred - prev_actual),
                    "actual_direction": np.sign(actual - prev_actual),
                    "lower80": float(ci80_arr[-1, 0]),
                    "upper80": float(ci80_arr[-1, 1]),
                    "lower95": float(ci95_arr[-1, 0]),
                    "upper95": float(ci95_arr[-1, 1]),
                }
            )

    return pd.DataFrame(rows)


def summarize_backtest(bt: pd.DataFrame) -> pd.DataFrame:
    if bt.empty:
        return pd.DataFrame()

    rows = []
    for (model_name, target, horizon), g in bt.groupby(["model", "target", "horizon"]):
        coverage80 = ((g["actual"] >= g["lower80"]) & (g["actual"] <= g["upper80"])).mean() * 100
        coverage95 = ((g["actual"] >= g["lower95"]) & (g["actual"] <= g["upper95"])).mean() * 100
        dir_acc = (g["pred_direction"] == g["actual_direction"]).mean() * 100

        rows.append(
            {
                "model": model_name,
                "target": target,
                "horizon": horizon,
                "n_forecasts": len(g),
                "RMSE": rmse(g["actual"], g["prediction"]),
                "MAE": mae(g["actual"], g["prediction"]),
                "MAPE": safe_mape(g["actual"], g["prediction"]),
                "Bias": float((g["prediction"] - g["actual"]).mean()),
                "Directional_Accuracy_pct": dir_acc,
                "Coverage80_pct": coverage80,
                "Coverage95_pct": coverage95,
                "AvgWidth80": float((g["upper80"] - g["lower80"]).mean()),
                "AvgWidth95": float((g["upper95"] - g["lower95"]).mean()),
            }
        )

    return pd.DataFrame(rows).sort_values(["target", "horizon", "RMSE"]).reset_index(drop=True)


# =========================================================
# 10) DIEBOLD-MARIANO TEST
# =========================================================
def diebold_mariano_test(
    y_true: Sequence[float],
    pred1: Sequence[float],
    pred2: Sequence[float],
    horizon: int = 1,
    power: int = 2,
) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    pred1 = np.asarray(pred1, dtype=float)
    pred2 = np.asarray(pred2, dtype=float)

    e1 = y_true - pred1
    e2 = y_true - pred2
    d = (np.abs(e1) ** power) - (np.abs(e2) ** power)

    T = len(d)
    if T < 5:
        return {"dm_stat": np.nan, "p_value": np.nan, "mean_loss_diff": np.nan}

    d_bar = np.mean(d)
    gamma0 = np.var(d, ddof=1)
    long_run_var = gamma0

    for lag in range(1, horizon):
        if T - lag <= 1:
            continue
        cov = np.cov(d[lag:], d[:-lag], ddof=1)[0, 1]
        long_run_var += 2 * (1 - lag / horizon) * cov

    if long_run_var <= 0:
        return {"dm_stat": np.nan, "p_value": np.nan, "mean_loss_diff": float(d_bar)}

    dm_stat = d_bar / np.sqrt(long_run_var / T)
    harvey_adj = np.sqrt((T + 1 - 2 * horizon + (horizon * (horizon - 1)) / T) / T)
    dm_stat_adj = dm_stat * harvey_adj
    p_val = 2 * (1 - stats.t.cdf(np.abs(dm_stat_adj), df=T - 1))

    return {
        "dm_stat": float(dm_stat_adj),
        "p_value": float(p_val),
        "mean_loss_diff": float(d_bar),
    }


def dm_table(bt_a: pd.DataFrame, bt_b: pd.DataFrame, label_a: str, label_b: str) -> pd.DataFrame:
    rows = []

    common_targets = sorted(set(bt_a["target"]).intersection(set(bt_b["target"])))
    for target in common_targets:
        common_horizons = sorted(
            set(bt_a.loc[bt_a["target"] == target, "horizon"]).intersection(
                set(bt_b.loc[bt_b["target"] == target, "horizon"])
            )
        )

        for h in common_horizons:
            a = bt_a[(bt_a["target"] == target) & (bt_a["horizon"] == h)][["forecast_date", "actual", "prediction"]].rename(columns={"prediction": "pred_a"})
            b = bt_b[(bt_b["target"] == target) & (bt_b["horizon"] == h)][["forecast_date", "prediction"]].rename(columns={"prediction": "pred_b"})
            m = a.merge(b, on="forecast_date", how="inner").dropna()

            if len(m) == 0:
                continue

            dm = diebold_mariano_test(m["actual"], m["pred_a"], m["pred_b"], horizon=int(h), power=2)

            rows.append(
                {
                    "target": target,
                    "horizon": h,
                    "model_A": label_a,
                    "model_B": label_b,
                    "DM_stat": dm["dm_stat"],
                    "DM_p_value": dm["p_value"],
                    "mean_loss_diff_A_minus_B": dm["mean_loss_diff"],
                }
            )

    return pd.DataFrame(rows)


# =========================================================
# 11) MAIN
# =========================================================
def main():
    project_root = find_project_root(PROJECT_ROOT_CANDIDATES)
    print("Project root:", project_root)

    merged_df = load_sentiment_merged(project_root)
    macro_df = load_macro_baseline(project_root)

    branch1_df = prepare_branch1_macro_df(macro_df)
    branch2_df = prepare_branch2_overlap_df(merged_df)

    print("=" * 100)
    print("A. DATA OVERVIEW")
    print("=" * 100)
    print("Merged sentiment+macro shape:", merged_df.shape)
    print("Macro-only shape:", macro_df.shape)
    print("Branch 2 actual overlap shape:", branch2_df.shape)
    print("Branch 2 date range:", branch2_df[DATE_COL].min(), "->", branch2_df[DATE_COL].max())

    print("\n" + "=" * 100)
    print("B. LEAKAGE AUDIT FOR SENTIMENT DATA")
    print("=" * 100)
    audit_df = leakage_audit(merged_df, SENTIMENT_AUDIT_VARS, SENTIMENT_REAL_START)
    print(audit_df.to_string(index=False))
    audit_df.to_csv(OUTPUT_DIR / "leakage_audit_pre_2019Q3.csv", index=False)

    print("\n" + "=" * 100)
    print("C. STATIONARITY (ADF/KPSS)")
    print("=" * 100)
    stationarity_vars = [
        "log_gdp",
        "log_gdp_diff1",
        "gdp_growth",
        "gdp_qoq_pct_recalc",
        "inflation",
        "interest_rate",
        "avg_sentiment",
        "net_sentiment",
        "n_comments",
    ]
    st_tbl = stationarity_table(merged_df, stationarity_vars)
    print(st_tbl.round(4).to_string(index=False))
    st_tbl.to_csv(OUTPUT_DIR / "stationarity_table_branch2.csv", index=False)

    print("\n" + "=" * 100)
    print("D. VIF FOR THE OVERLAP MODEL")
    print("=" * 100)
    vif_tbl = vif_table(branch2_df, MACRO_EXOG + SENTIMENT_MODEL_VARS)
    print(vif_tbl.round(4).to_string(index=False))
    vif_tbl.to_csv(OUTPUT_DIR / "vif_overlap_branch2.csv", index=False)

    print("\n" + "=" * 100)
    print("E. FINAL MACRO SARIMAX BASELINE")
    print("=" * 100)
    macro_order = ORDER_MAP[TARGET_MACRO]["macro_final"]
    _, macro_res = fit_sarimax_full(
        df=branch1_df,
        target=TARGET_MACRO,
        order=macro_order,
        exog_cols=MACRO_EXOG,
    )
    print(macro_res.summary())

    macro_coef_tbl = coefficient_table(macro_res)
    print("\nFinal macro SARIMAX coefficient table:")
    print(macro_coef_tbl.round(4).to_string(index=False))
    macro_coef_tbl.to_csv(OUTPUT_DIR / "final_macro_sarimax_coefficients.csv", index=False)

    macro_diag = residual_diagnostics(macro_res, lb_lags=(4, 8), arch_lags=4)
    print("\nLjung-Box residual test:")
    print(macro_diag["ljung_box"].round(4).to_string(index=False))
    print("\nResidual overall diagnostics:")
    print(macro_diag["overall"].round(4).to_string(index=False))
    macro_diag["ljung_box"].to_csv(OUTPUT_DIR / "final_macro_ljungbox.csv", index=False)
    macro_diag["overall"].to_csv(OUTPUT_DIR / "final_macro_residual_overall.csv", index=False)

    print("\n" + "=" * 100)
    print("F. DYNAMIC REGRESSION ON THE ACTUAL OVERLAP")
    print("=" * 100)
    overlap_reg = branch2_df.copy()
    if "gdp_growth_lag1" not in overlap_reg.columns:
        overlap_reg["gdp_growth_lag1"] = make_lag(overlap_reg, "gdp_growth", 1)

    _, macro_plain, macro_hac = fit_dynamic_regression(
        df=overlap_reg,
        target="gdp_growth",
        predictors=["gdp_growth_lag1", "inflation", "interest_rate"],
        hac_lags=1,
    )

    _, sent_plain, sent_hac = fit_dynamic_regression(
        df=overlap_reg,
        target="gdp_growth",
        predictors=["gdp_growth_lag1", "inflation", "interest_rate"] + SENTIMENT_MODEL_VARS,
        hac_lags=1,
    )

    compare_tbl = pd.DataFrame(
        {
            "model": ["Macro-only dynamic regression", "Macro + sentiment dynamic regression"],
            "Adj_R2": [macro_plain.rsquared_adj, sent_plain.rsquared_adj],
            "AICc": [aicc_from_result(macro_plain), aicc_from_result(sent_plain)],
        }
    )
    print(compare_tbl.round(4).to_string(index=False))
    compare_tbl.to_csv(OUTPUT_DIR / "dynamic_regression_compare.csv", index=False)

    macro_coef = coefficient_table(macro_hac)
    sent_coef = coefficient_table(sent_hac)

    print("\nMacro-only dynamic regression coefficients:")
    print(macro_coef.round(4).to_string(index=False))
    print("\nMacro + sentiment dynamic regression coefficients:")
    print(sent_coef.round(4).to_string(index=False))

    macro_coef.to_csv(OUTPUT_DIR / "dynamic_reg_macro_only_coef.csv", index=False)
    sent_coef.to_csv(OUTPUT_DIR / "dynamic_reg_macro_sent_coef.csv", index=False)

    nested_tbl = nested_sentiment_tests(
        res_small_plain=macro_plain,
        res_big_plain=sent_plain,
        res_big_hac=sent_hac,
        sentiment_terms=SENTIMENT_MODEL_VARS,
    )
    print("\nNested / joint sentiment tests:")
    print(nested_tbl.round(6).to_string(index=False))
    nested_tbl.to_csv(OUTPUT_DIR / "nested_sentiment_tests.csv", index=False)

    print("\n" + "=" * 100)
    print("G. CHOW TEST / STRUCTURAL BREAK")
    print("=" * 100)
    chow_rows = []
    for br in BREAK_DATES:
        out = chow_test(
            df=overlap_reg,
            target="gdp_growth",
            predictors=["gdp_growth_lag1", "inflation", "interest_rate"] + SENTIMENT_MODEL_VARS,
            break_date=br,
        )
        chow_rows.append(out)

    chow_tbl = pd.DataFrame(chow_rows)
    print(chow_tbl.round(6).to_string(index=False))
    chow_tbl.to_csv(OUTPUT_DIR / "chow_break_tests_branch2.csv", index=False)

    print("\n" + "=" * 100)
    print("H. MULTI-HORIZON ROLLING-ORIGIN BACKTEST")
    print("=" * 100)
    all_backtests = []

    for tgt in TARGET_OVERLAPS:
        print(f"\n--- TARGET: {tgt} ---")

        bt_macro = rolling_origin_backtest(
            df=branch2_df,
            target=tgt,
            order=ORDER_MAP[tgt]["macro_overlap"],
            exog_cols=MACRO_EXOG,
            initial_train_end=TRAIN_END,
            horizons=FORECAST_HORIZONS,
            model_name=f"{tgt}_macro_only",
        )

        bt_macro_sent = rolling_origin_backtest(
            df=branch2_df,
            target=tgt,
            order=ORDER_MAP[tgt]["macro_sent_overlap"],
            exog_cols=MACRO_EXOG + SENTIMENT_MODEL_VARS,
            initial_train_end=TRAIN_END,
            horizons=FORECAST_HORIZONS,
            model_name=f"{tgt}_macro_plus_sentiment",
        )

        all_backtests.extend([bt_macro, bt_macro_sent])

        summary_tgt = summarize_backtest(pd.concat([bt_macro, bt_macro_sent], ignore_index=True))
        dm_tgt = dm_table(
            bt_a=bt_macro,
            bt_b=bt_macro_sent,
            label_a=f"{tgt}_macro_only",
            label_b=f"{tgt}_macro_plus_sentiment",
        )

        print("\nBacktest summary:")
        print(summary_tgt.round(4).to_string(index=False))
        summary_tgt.to_csv(OUTPUT_DIR / f"backtest_summary_{tgt}.csv", index=False)

        print("\nDiebold-Mariano test:")
        if len(dm_tgt) > 0:
            print(dm_tgt.round(6).to_string(index=False))
        else:
            print("Not enough data to run the DM test.")
        dm_tgt.to_csv(OUTPUT_DIR / f"dm_test_{tgt}.csv", index=False)

        bt_macro.to_csv(OUTPUT_DIR / f"backtest_detail_{tgt}_macro_only.csv", index=False)
        bt_macro_sent.to_csv(OUTPUT_DIR / f"backtest_detail_{tgt}_macro_plus_sentiment.csv", index=False)

    if len(all_backtests) > 0:
        all_bt = pd.concat(all_backtests, ignore_index=True)
        all_bt.to_csv(OUTPUT_DIR / "all_backtests_overlap.csv", index=False)

    print("\n" + "=" * 100)
    print("I. SEPARATE BACKTEST FOR THE FINAL MACRO BASELINE")
    print("=" * 100)
    bt_final_macro = rolling_origin_backtest(
        df=branch1_df,
        target=TARGET_MACRO,
        order=ORDER_MAP[TARGET_MACRO]["macro_final"],
        exog_cols=MACRO_EXOG,
        initial_train_end=TRAIN_END,
        horizons=FORECAST_HORIZONS,
        model_name="final_macro_baseline",
    )

    final_summary = summarize_backtest(bt_final_macro)
    print(final_summary.round(4).to_string(index=False))

    bt_final_macro.to_csv(OUTPUT_DIR / "backtest_detail_final_macro_baseline.csv", index=False)
    final_summary.to_csv(OUTPUT_DIR / "backtest_summary_final_macro_baseline.csv", index=False)

    summary_notes = pd.DataFrame(
        {
            "item": [
                "project_root",
                "merged_sentiment_file",
                "macro_file",
                "sentiment_real_start",
                "train_end",
                "forecast_horizons",
                "target_macro",
                "target_overlaps",
            ],
            "value": [
                str(project_root),
                str(project_root / MERGED_SENTIMENT_FILE_REL),
                str(project_root / MACRO_BASELINE_FILE_REL),
                SENTIMENT_REAL_START,
                TRAIN_END,
                str(FORECAST_HORIZONS),
                TARGET_MACRO,
                str(TARGET_OVERLAPS),
            ],
        }
    )
    summary_notes.to_csv(OUTPUT_DIR / "run_metadata.csv", index=False)

    print("\nDone. All result tables have been saved in the folder:")
    print(OUTPUT_DIR.resolve())


if __name__ == "__main__":
    main()
